# Clear Cost Delta Files

Drops all Lakehouse tables created by `01_download_cost_data` and removes staging CSVs.

**Tables dropped:** `cost_record`, `subscription`, `resource_group`, `resource`, `meter_category`, `service`

**Prerequisites:**
- Attach the same Lakehouse used by `01_download_cost_data`

In [ ]:
import os, glob, traceback
from datetime import datetime

TABLES = ["cost_record", "subscription", "resource_group", "resource", "meter_category", "service"]
CSV_DIR = "/lakehouse/default/Files/cost_daily"

print(f"Clear started: {datetime.utcnow().isoformat()}Z")

# Drop Delta tables
for t in TABLES:
    try:
        if spark.catalog.tableExists(t):
            before = spark.table(t).count()
            spark.sql(f"DROP TABLE IF EXISTS {t}")
            print(f"  {t}: DROPPED ({before:,} rows)")
        else:
            print(f"  {t}: not found")
    except Exception as e:
        print(f"  {t}: ERROR - {e}")
        traceback.print_exc()

# Clear staging CSVs
removed = 0
try:
    if os.path.isdir(CSV_DIR):
        for f in glob.glob(os.path.join(CSV_DIR, "cost_*.csv")):
            os.remove(f)
            removed += 1
except Exception as e:
    print(f"  CSV cleanup error: {e}")
print(f"  Cleared {removed} staging CSV(s) from {CSV_DIR}")

print(f"Clear completed: {datetime.utcnow().isoformat()}Z")